# Dropout Demo — Section 6.3

**Dropout** randomly sets neurons to zero during training with probability **p** (the dropout rate).

This prevents neurons from co-adapting and acts as a regularizer — equivalent to training an ensemble of $2^N$ sub-networks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets


def draw_dropout(H=5, L=2, dropout_rate=0.5, seed=42):
    rng = np.random.default_rng(seed)
    layers = [2] + [H] * L + [1]
    total_layers = len(layers)

    # Determine which neurons are dropped
    dropped = {}
    for li in range(1, total_layers - 1):  # hidden layers only
        mask = rng.random(layers[li]) < dropout_rate
        dropped[li] = mask

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    titles = ['Full network (no dropout)', f'During training (p={dropout_rate:.0%} dropout)']

    for ax_idx, ax in enumerate(axes):
        ax.set_xlim(-0.5, total_layers - 0.5)
        ax.set_ylim(-0.5, max(layers) - 0.5)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(titles[ax_idx], fontsize=12)

        pos = {}  # (layer, idx) -> (x, y)
        for li, n in enumerate(layers):
            offset = (max(layers) - n) / 2
            for ni in range(n):
                pos[(li, ni)] = (li, offset + ni)

        # Draw connections
        for li in range(total_layers - 1):
            for ni in range(layers[li]):
                for nj in range(layers[li + 1]):
                    if ax_idx == 1:
                        d1 = li in dropped and dropped[li][ni]
                        d2 = (li+1) in dropped and dropped[li+1][nj]
                        if d1 or d2:
                            continue
                    x1, y1 = pos[(li, ni)]
                    x2, y2 = pos[(li+1, nj)]
                    ax.plot([x1, x2], [y1, y2], color='#cbd5e1', lw=0.8, zorder=1)

        # Draw neurons
        for li, n in enumerate(layers):
            for ni in range(n):
                x, y = pos[(li, ni)]
                is_dropped = ax_idx == 1 and li in dropped and dropped[li][ni]
                if li == 0:
                    color, edge = '#86efac', '#059669'
                elif li == total_layers - 1:
                    color, edge = '#fca5a5', '#dc2626'
                elif is_dropped:
                    color, edge = '#f1f5f9', '#94a3b8'
                else:
                    color, edge = '#bfdbfe', '#2563eb'
                circle = plt.Circle((x, y), 0.3, fc=color, ec=edge, lw=2, zorder=2)
                ax.add_patch(circle)
                if is_dropped:
                    ax.plot([x-0.15, x+0.15], [y-0.15, y+0.15], color='#94a3b8', lw=2, zorder=3)
                    ax.plot([x+0.15, x-0.15], [y-0.15, y+0.15], color='#94a3b8', lw=2, zorder=3)

    total_hidden = H * L
    dropped_count = sum(m.sum() for m in dropped.values())
    fig.suptitle(
        f'Hidden neurons: {total_hidden}  |  Dropped: {dropped_count} ({dropped_count/total_hidden:.0%})  |  Active: {total_hidden-dropped_count}',
        fontsize=11, y=0.02
    )
    plt.tight_layout()
    plt.show()


widgets.interact(
    draw_dropout,
    H=widgets.IntSlider(min=3, max=8, step=1, value=5, description='Neurons/layer', continuous_update=False),
    L=widgets.IntSlider(min=1, max=3, step=1, value=2, description='Hidden layers', continuous_update=False),
    dropout_rate=widgets.FloatSlider(min=0, max=0.9, step=0.1, value=0.5, description='Dropout p', continuous_update=False),
    seed=widgets.IntSlider(min=1, max=20, step=1, value=42, description='Seed', continuous_update=False),
)